# Pilot: where should the LLM sit in an automated underwriting pipeline?

This notebook reproduces the pilot end to end on Kaggle: it cleans the Give Me Some Credit data, trains the scorecard,
builds synthetic document bundles for 200 applicants and a subset of 8 attacks, runs one small open-weight model on
architectures A, A with a prompt defense, and the extractor used by B and C, and prints the main metrics.

Requirements: a GPU session (T4 x2) with internet on, and the dataset `brycecf/give-me-some-credit-dataset` attached.
Expected runtime: about 15 minutes. All data and payloads are synthetic.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/craigouma/llm-underwriting-injection"
REPO = Path(os.environ.get("UW_REPO_DIR", "/kaggle/working/repo"))
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
RAW = next(Path("/kaggle/input").rglob("cs-training.csv"))
(REPO / "data/raw").mkdir(parents=True, exist_ok=True)
if not (REPO / "data/raw/cs-training.csv").exists():
    (REPO / "data/raw/cs-training.csv").symlink_to(RAW)
print("repo", REPO, "data", RAW)

## 1. Clean the data, write the splits and train the scorecards

In [ ]:
!python -m src.train_scorecard | tail -n 12

## 2. Run the model on the pilot applicants (raw outputs are cached to `results/raw/pilot`)

In [ ]:
!python kaggle/infer/infer.py --model Qwen/Qwen2.5-1.5B-Instruct --run pilot

## 3. Replay the cached outputs through every architecture and summarise

In [ ]:
!python -m src.evaluate --run pilot --models Qwen/Qwen2.5-1.5B-Instruct --oracle

In [ ]:
import pandas as pd
from src import analysis as an
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
df = an.with_oracle(an.load("pilot"))
display(an.clean_utility(df).round(3))
display(an.asr(df, ["category"])[["model", "arm", "category", "k", "n", "rate", "lo", "hi"]].round(3))